# 02 — Mobility-weighted destination PM₂.₅ exposure in Helsinki

**Purpose.** Extend the residential baseline using CAFEIN weekday OD flows. PM₂.₅ is sampled for each neighbourhood, attached to OD destinations, and aggregated into a trip-weighted destination-exposure indicator for each origin neighbourhood.

This notebook also compares origin PM₂.₅ with mobility-weighted destination PM₂.₅. It stops **before** route generation; route exposure is isolated in Notebook 03.


In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt

import cafein.sampledata.helsinki as helsinki

print("Air quality:", helsinki.air_quality)
print("OD weekday:", helsinki.mobility.od_weekday)
print("Presence weekday:", helsinki.mobility.presence_weekday)
print("Neighbourhoods:", helsinki.mobility.neighbourhoods)

## 1. Load mobility and neighbourhood data

In [ ]:
od = pd.read_parquet(helsinki.mobility.od_weekday)
presence = pd.read_parquet(helsinki.mobility.presence_weekday)
units = gpd.read_file(helsinki.mobility.neighbourhoods)

print("OD:", od.shape, od.columns.tolist())
print("Presence:", presence.shape, presence.columns.tolist())
print("Units:", units.shape, units.columns.tolist(), units.crs)

display(od.head())
display(presence.head())
display(units.head())

## 2. Normalize neighbourhood IDs

In [ ]:
units["unit_id"] = units["Neighbourhood_unit_ID"].astype("Int64")

presence["home_unit"] = presence["home_unit"].astype("Int64")
presence["visited_unit"] = presence["visited_unit"].astype("Int64")

od["origin"] = od["origin"].astype("Int64")
od["destination"] = od["destination"].astype("Int64")

unit_lookup = units[
    ["unit_id", "Neighbourhood_name", "Municipality_name", "geometry"]
].copy()

## 3. Add readable neighbourhood names to OD flows

In [ ]:
origin_names = unit_lookup[
    ["unit_id", "Neighbourhood_name"]
].rename(columns={
    "unit_id": "origin",
    "Neighbourhood_name": "origin_name"
})

destination_names = unit_lookup[
    ["unit_id", "Neighbourhood_name"]
].rename(columns={
    "unit_id": "destination",
    "Neighbourhood_name": "destination_name"
})

od_geo = (
    od
    .merge(origin_names, on="origin", how="left")
    .merge(destination_names, on="destination", how="left")
)

display(od_geo.head())

print("Missing origin names:", od_geo["origin_name"].isna().sum())
print("Missing destination names:", od_geo["destination_name"].isna().sum())

## 4. Inspect the FMI-ENFUSER PM₂.₅ raster

Band 5 is PM₂.₅. The sample raster has no declared NoData value, so out-of-bounds coordinates must be handled explicitly.

In [ ]:
with rasterio.open(helsinki.air_quality) as src:
    print("CRS:", src.crs)
    print("NoData:", src.nodata)
    print("Bounds:", src.bounds)
    print("Band 5:", src.descriptions[4])

    pm = src.read(5, masked=True)
    print("Valid min:", float(pm.min()))
    print("Valid max:", float(pm.max()))
    print("Valid mean:", float(pm.mean()))

## 5. Sample PM₂.₅ at one representative point per neighbourhood

In [ ]:
unit_pm = units[
    ["unit_id", "Neighbourhood_name", "Municipality_name", "geometry"]
].copy()

with rasterio.open(helsinki.air_quality) as src:
    # Put polygons in the raster CRS (EPSG:4326 for this sample)
    unit_pm = unit_pm.to_crs(src.crs)

    # Guaranteed to lie inside each polygon
    unit_pm["sample_point"] = unit_pm.geometry.representative_point()

    # Raster bounds can have reversed y ordering, so normalize them
    xmin = min(src.bounds.left, src.bounds.right)
    xmax = max(src.bounds.left, src.bounds.right)
    ymin = min(src.bounds.bottom, src.bounds.top)
    ymax = max(src.bounds.bottom, src.bounds.top)

    def sample_pm25(point):
        x, y = point.x, point.y

        # Explicit out-of-bounds protection
        if not (xmin <= x <= xmax and ymin <= y <= ymax):
            return np.nan

        value = next(
            src.sample([(x, y)], indexes=5, masked=True)
        )[0]

        if np.ma.is_masked(value):
            return np.nan

        value = float(value)

        # Safety check based on the actual raster: zero is not a valid value here
        if value <= 0:
            return np.nan

        return value

    unit_pm["pm25"] = unit_pm["sample_point"].apply(sample_pm25)

print("Minimum:", unit_pm["pm25"].min())
print("Maximum:", unit_pm["pm25"].max())
print("Missing neighbourhoods:", unit_pm["pm25"].isna().sum())

display(
    unit_pm[
        ["unit_id", "Neighbourhood_name", "Municipality_name", "pm25"]
    ].head(20)
)

In [ ]:
missing_units = unit_pm.loc[
    unit_pm["pm25"].isna(),
    ["unit_id", "Neighbourhood_name", "Municipality_name"]
]

display(missing_units)

## 6. Attach destination PM₂.₅ to OD flows

This is a **destination-exposure indicator**, not yet exposure during travel.

In [ ]:
destination_pm = unit_pm[
    ["unit_id", "pm25"]
].rename(columns={
    "unit_id": "destination",
    "pm25": "destination_pm25"
})

od_exposure = od_geo.merge(
    destination_pm,
    on="destination",
    how="left"
)

valid_od = od_exposure[
    od_exposure["destination_pm25"].notna()
].copy()

valid_od["exposure_weight"] = (
    valid_od["trips"] * valid_od["destination_pm25"]
)

print("OD rows:", len(od_exposure))
print("OD rows with valid destination PM2.5:", len(valid_od))

## 7. Trip-weighted destination PM₂.₅ and environmental coverage

In [ ]:
origin_exposure = (
    valid_od
    .groupby(["origin", "origin_name"], as_index=False)
    .agg(
        valid_trips=("trips", "sum"),
        exposure_weight=("exposure_weight", "sum")
    )
)

origin_exposure["trip_weighted_destination_pm25"] = (
    origin_exposure["exposure_weight"] /
    origin_exposure["valid_trips"]
)

total_trips = (
    od_geo
    .groupby(["origin", "origin_name"], as_index=False)
    .agg(total_trips=("trips", "sum"))
)

origin_exposure = origin_exposure.merge(
    total_trips,
    on=["origin", "origin_name"],
    how="left"
)

origin_exposure["coverage_pct"] = (
    100 * origin_exposure["valid_trips"] /
    origin_exposure["total_trips"]
)

display(
    origin_exposure[
        [
            "origin_name",
            "trip_weighted_destination_pm25",
            "coverage_pct",
            "total_trips"
        ]
    ]
    .sort_values("trip_weighted_destination_pm25", ascending=False)
    .head(20)
)

display(origin_exposure["coverage_pct"].describe())

## 8. Compare origin PM₂.₅ with trip-weighted destination PM₂.₅

In [ ]:
origin_pm = unit_pm[
    ["unit_id", "pm25"]
].rename(columns={
    "unit_id": "origin",
    "pm25": "origin_pm25"
})

origin_exposure = origin_exposure.merge(
    origin_pm,
    on="origin",
    how="left"
)

origin_exposure["mobility_exposure_change"] = (
    origin_exposure["trip_weighted_destination_pm25"] -
    origin_exposure["origin_pm25"]
)

comparison = origin_exposure[
    origin_exposure["origin_pm25"].notna()
].copy()

display(
    comparison[
        [
            "origin_name",
            "origin_pm25",
            "trip_weighted_destination_pm25",
            "mobility_exposure_change",
            "coverage_pct",
            "total_trips"
        ]
    ]
    .sort_values("mobility_exposure_change", ascending=False)
    .head(20)
)

## 9. Map trip-weighted destination PM₂.₅

In [ ]:
map_exposure = units.merge(
    origin_exposure[
        ["origin", "trip_weighted_destination_pm25", "coverage_pct"]
    ],
    left_on="unit_id",
    right_on="origin",
    how="left"
)

fig, ax = plt.subplots(figsize=(11, 10))

map_exposure.plot(
    column="trip_weighted_destination_pm25",
    legend=True,
    ax=ax,
    missing_kwds={"label": "No data"}
)

ax.set_title("Trip-weighted destination PM₂.₅ by origin neighbourhood")
ax.set_axis_off()

plt.tight_layout()
plt.show()

### Interpretation guardrails

- `trip_weighted_destination_pm25` is a **destination-exposure indicator**, not exposure during travel.
- `mobility_exposure_change` compares the PM₂.₅ sampled at the origin neighbourhood with the PM₂.₅ of destinations weighted by trip volume.
- The CAFEIN sample OD table has hour-of-day information but is not a date-specific event panel here, so this notebook does **not** establish behavioural response to a smoke or heat event.
- The `presence` table is loaded as in the original workflow but is not yet used in this analysis; it is a natural next extension for activity-space exposure.
